# JEPA-for-Trading V1

Kaggle runner for a multi-asset JEPA market model plus PPO portfolio agent.

In [ ]:
# Run this cell on Kaggle if the repo is not already available.
import os
from pathlib import Path

if not Path('jepa-for-trading').exists():
    !git clone -b version1 https://github.com/aurvl/jepa-for-trading.git
%cd jepa-for-trading
!pip install -q -e .

In [ ]:
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from jepa_trading.config import ensure_dirs, load_config
from jepa_trading.data.pipeline import prepare_market_data, create_jepa_dataloaders
from jepa_trading.models.jepa import MarketJEPA
from jepa_trading.models.heads import MarketHeads
from jepa_trading.models.policy import PortfolioPolicy
from jepa_trading.rl.observer import JEPAMarketObserver
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.ppo import PPOConfig, train_ppo
from jepa_trading.training.train_jepa import train_jepa
from jepa_trading.training.train_heads import train_market_heads
from jepa_trading.training.checkpoints import load_checkpoint, save_checkpoint
from jepa_trading.evaluation.backtest import (
    run_policy_backtest,
    run_weight_strategy,
    buy_and_hold_weight,
    equal_weight,
    momentum_weight,
    volatility_target_weight,
    random_long_only_weight,
)
from jepa_trading.evaluation.metrics import metrics_table
from jepa_trading.evaluation.plots import plot_equity_curves, plot_drawdown, plot_turnover
from jepa_trading.evaluation.statistical_tests import randomization_p_value, bootstrap_mean_return_p_value
from jepa_trading.utils.device import get_device
from jepa_trading.utils.seed import seed_everything

In [ ]:
config = load_config('configs/default.yaml')

# Kaggle: upload macro_data.parquet as a dataset. This cell auto-discovers it.
macro_candidates = []
if Path('/kaggle/input').exists():
    macro_candidates = list(Path('/kaggle/input').rglob('macro_data.parquet'))
if macro_candidates:
    config['data']['macro_path'] = str(macro_candidates[0])

# Toggle for debugging. Full run uses the YAML values.
FAST_DEV_RUN = False
if FAST_DEV_RUN:
    config['data']['tickers'] = config['data']['tickers'][:8]
    config['training']['jepa_max_steps'] = 20
    config['training']['heads_max_steps'] = 20
    config['training']['eval_every'] = 10
    config['ppo']['total_updates'] = 2
    config['ppo']['rollout_steps'] = 32

ensure_dirs(config)
seed_everything(config['seed'])
device = get_device(config['device'])
device

In [ ]:
prepared_df, arrays, feature_columns = prepare_market_data(config, force_download=False)
loaders = create_jepa_dataloaders(config, arrays)

print('rows:', len(prepared_df))
print('dates:', arrays.dates.min(), '->', arrays.dates.max())
print('assets:', len(arrays.tickers), arrays.tickers)
print('features:', len(feature_columns), feature_columns)
print('dataset sizes:', {k: len(v.dataset) for k, v in loaders.items()})

In [ ]:
jepa = MarketJEPA(
    n_features=len(feature_columns),
    max_assets=len(arrays.tickers),
    **config['model'],
    ema_decay=config['training']['ema_decay'],
)
jepa_ckpt = Path(config['training']['checkpoint_dir']) / 'market_jepa.pt'

if jepa_ckpt.exists():
    print('Loading JEPA checkpoint:', jepa_ckpt)
    load_checkpoint(jepa_ckpt, jepa, map_location=device)
    jepa_history = pd.DataFrame()
else:
    jepa_history = train_jepa(
        jepa,
        loaders['train'],
        loaders['val'],
        device,
        max_steps=config['training']['jepa_max_steps'],
        lr=config['training']['lr'],
        weight_decay=config['training']['weight_decay'],
        checkpoint_path=jepa_ckpt,
        eval_every=config['training']['eval_every'],
        log_every=config['training']['log_every'],
    )
    load_checkpoint(jepa_ckpt, jepa, map_location=device)
jepa_history.tail()

In [ ]:
heads = MarketHeads(latent_dim=config['model']['latent_dim'])
heads_ckpt = Path(config['training']['checkpoint_dir']) / 'market_heads.pt'

if heads_ckpt.exists():
    print('Loading heads checkpoint:', heads_ckpt)
    load_checkpoint(heads_ckpt, heads, map_location=device)
    heads_history = pd.DataFrame()
else:
    heads_history = train_market_heads(
        jepa,
        heads,
        loaders['train'],
        loaders['val'],
        device,
        max_steps=config['training']['heads_max_steps'],
        lr=config['training']['lr'],
        checkpoint_path=heads_ckpt,
        eval_every=config['training']['eval_every'],
    )
    load_checkpoint(heads_ckpt, heads, map_location=device)
heads_history.tail()

In [ ]:
observer = JEPAMarketObserver(
    arrays=arrays,
    jepa=jepa,
    heads=heads,
    lookback=config['data']['lookback'],
    horizons=config['data']['horizons'],
    device=device,
)

train_env = TradingEnv(
    arrays,
    observer,
    start_date=pd.Timestamp(config['data']['train_end']) - pd.DateOffset(years=3),
    end_date=pd.Timestamp(config['data']['val_end']),
    lookback=config['data']['lookback'],
    cash_initial=config['portfolio']['cash_initial'],
    transaction_cost_bps=config['portfolio']['transaction_cost_bps'],
    max_weight_per_asset=config['portfolio']['max_weight_per_asset'],
    max_turnover=config['portfolio']['max_turnover'],
    drawdown_penalty=config['ppo']['reward_drawdown_penalty'],
    turnover_penalty=config['ppo']['reward_turnover_penalty'],
    concentration_penalty=config['ppo']['reward_concentration_penalty'],
)
obs0, mask0 = train_env.reset()
policy = PortfolioPolicy(
    obs_dim=len(obs0),
    n_assets=len(arrays.tickers),
    max_weight=config['portfolio']['max_weight_per_asset'],
)
ppo_cfg = PPOConfig(**{k: config['ppo'][k] for k in PPOConfig.__dataclass_fields__.keys()})
policy_ckpt = Path(config['training']['checkpoint_dir']) / 'ppo_policy.pt'

if policy_ckpt.exists():
    print('Loading PPO policy:', policy_ckpt)
    load_checkpoint(policy_ckpt, policy, map_location=device)
    ppo_history = pd.DataFrame()
else:
    ppo_history = train_ppo(train_env, policy, ppo_cfg, device)
    save_checkpoint(policy_ckpt, policy, kind='ppo_policy')
ppo_history.tail()

In [ ]:
test_start = pd.Timestamp(config['data']['val_end']) + pd.Timedelta(days=1)
test_end = arrays.dates.max()

def make_test_env():
    return TradingEnv(
        arrays,
        observer,
        start_date=test_start,
        end_date=test_end,
        lookback=config['data']['lookback'],
        cash_initial=config['portfolio']['cash_initial'],
        transaction_cost_bps=config['portfolio']['transaction_cost_bps'],
        max_weight_per_asset=config['portfolio']['max_weight_per_asset'],
        max_turnover=config['portfolio']['max_turnover'],
        drawdown_penalty=config['ppo']['reward_drawdown_penalty'],
        turnover_penalty=config['ppo']['reward_turnover_penalty'],
        concentration_penalty=config['ppo']['reward_concentration_penalty'],
    )

agent_hist = run_policy_backtest(make_test_env(), policy, device)
bh_hist = run_weight_strategy(make_test_env(), buy_and_hold_weight)
equal_hist = run_weight_strategy(make_test_env(), equal_weight)
mom_hist = run_weight_strategy(make_test_env(), momentum_weight)
vol_hist = run_weight_strategy(make_test_env(), volatility_target_weight)

rng = np.random.default_rng(config['seed'])
random_hists = [
    run_weight_strategy(make_test_env(), lambda env, mask, rng=rng: random_long_only_weight(env, mask, rng))
    for _ in range(100)
]

histories = {
    'JEPA-PPO Agent': agent_hist,
    'Buy & Hold': bh_hist,
    'Equal Weight': equal_hist,
    'Momentum': mom_hist,
    'Vol Target': vol_hist,
}
metrics_table(histories)

In [ ]:
plot_equity_curves(
    agent_hist,
    bh_hist,
    random_hists,
    extra={'Equal Weight': equal_hist, 'Momentum': mom_hist, 'Vol Target': vol_hist},
)
plot_drawdown(agent_hist)
plot_turnover(agent_hist)

In [ ]:
rand_test = randomization_p_value(agent_hist, random_hists)
boot_test = bootstrap_mean_return_p_value(agent_hist, bh_hist)
print('Randomization test:', rand_test)
print('Bootstrap vs Buy & Hold:', boot_test)

if rand_test['p_value_random_beats_agent'] < 0.05:
    print('Interpretation: agent beats random strategies with strong empirical significance.')
elif rand_test['p_value_random_beats_agent'] < 0.20:
    print('Interpretation: agent is above most random strategies, but evidence is moderate.')
else:
    print('Interpretation: agent may still be close to random. Inspect reward, costs, splits, and overfitting.')